# Gauss divergence theorem on a torus

This tutorial demonstrates surface and volume integration with LowLevelFEM by verifying the Gauss divergence theorem

$$\int_{\partial V} \mathbf{v}\cdot\mathbf{n}\,\mathrm{d}A = \int_V \nabla\cdot\mathbf{v}\,\mathrm{d}V.$$

The domain is a solid torus with major radius $R$ and minor radius $r$. The surface normal, vector fields, divergence, and numerical integration are all evaluated through the standard LowLevelFEM interface.

## Load the package and initialize Gmsh

LowLevelFEM uses Gmsh for geometry, meshing, numerical integration rules, and post-processing.

In [1]:
using LowLevelFEM

gmsh.initialize()

## Create the torus mesh

The parameters are passed to `torus.geo` before the file is merged. The geometry file generates a third-order tetrahedral mesh and defines two physical groups: `"volu"` for the solid torus and `"surf"` for its closed boundary.

In [2]:
r = 5.0  # Minor radius of the torus
R = 10.0 # Major radius of the torus

setParameter("r", r)
setParameter("R", R)
gmsh.merge("torus.geo")

Info    : Reading 'torus.geo'...
Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Circle)
Info    : [ 60%] Meshing curve 2 (Circle)
Info    : Done meshing 1D (Wall 0.000354211s, CPU 0.00034s)
Info    : Meshing 2D...
Info    : Meshing surface 1 (Torus, Frontal-Delaunay)
Info    : Done meshing 2D (Wall 0.143725s, CPU 0.141682s)
Info    : Meshing 3D...
Info    : 3D Meshing 1 volume with 1 connected component
Info    : Tetrahedrizing 2423 nodes...
Info    : Done tetrahedrizing 2431 nodes (Wall 0.0491328s, CPU 0.047946s)
Info    : Reconstructing mesh...
Info    :  - Creating surface mesh
Info    :  - Identifying boundary edges
Info    :  - Recovering boundary
Info    : Done reconstructing mesh (Wall 0.123464s, CPU 0.118068s)
Info    : Found volume 1
Info    : It. 0 - 0 nodes created - worst tet radius 5.4174 (nodes removed 0 0)
Info    : It. 500 - 500 nodes created - worst tet radius 1.61852 (nodes removed 0 0)
Info    : It. 1000 - 1000 nodes created - worst tet radius 1.30217 (nod

## Construct a LowLevelFEM problem

No finite element equation is solved in this example. The `Problem` object provides the mesh and geometric information required by the field and integration operations.

In [3]:
mat = material("volu")
prob = Problem([mat]);

## Define the vector field

We use a smooth, nontrivial three-dimensional field so that neither side of the divergence theorem vanishes. Each component is written as an ordinary Julia function of the spatial coordinates.

In [4]:
vx(x, y, z) = x * y * z / 100
vy(x, y, z) = (x^2 + y^2 + z^2) / 10
vz(x, y, z) = (2x * y + 2y * z + 2z) / 10;

The outward unit normal is computed on the boundary elements. The same analytical vector field is then sampled separately on the boundary and in the volume because the two integrals act on different physical groups.

In [5]:
n = normalVector(prob, "surf")
v_volu = VectorField(prob, "volu", [vx, vy, vz])
v_surf = elementsToElements(v_volu, onPhysicalGroup="surf");
# or
# v_surf = VectorField(prob, "surf", [vx, vy, vz])

## Verify the divergence theorem

The left-hand side is the flux of $\mathbf{v}$ through the closed surface. The right-hand side is the volume integral of the divergence, obtained with the symbolic divergence operator `∇ ⋅`.

In [6]:
surface_flux = integrate(prob, "surf", v_surf ⋅ n)
volume_divergence = integrate(prob, "volu", ∇ ⋅ v_volu)
relative_difference = abs(surface_flux - volume_divergence) / abs(volume_divergence)

(; surface_flux, volume_divergence, relative_difference)

(surface_flux = 986.9573727384928, volume_divergence = 986.9638362043719, relative_difference = 6.548837598658074e-6)

The two numerical values should agree up to the discretization and quadrature errors. With the supplied third-order mesh, the relative difference is typically only a few parts in one million.

## Integrate constants: surface area and volume

Integrating the constant function $1$ measures the size of the selected physical group. For a torus,

$$A = 4\pi^2 Rr, \qquad V = 2\pi^2 Rr^2.$$

In [7]:
surface_area = integrate(prob, "surf", (x, y, z) -> 1.0)
exact_surface_area = 4π^2 * R * r

volume = integrate(prob, "volu", (x, y, z) -> 1.0)
exact_volume = 2π^2 * R * r^2

(; surface_area, exact_surface_area, volume, exact_volume)

(surface_area = 1973.9176714353496, exact_surface_area = 1973.9208802178719, volume = 4934.817665058359, exact_volume = 4934.802200544679)

## Inspect the fields in the Gmsh post-processor

The following views show the unit normal, the vector field, its normal component, and its divergence. `showElementResults` stores elementwise values as Gmsh post-processing views.

In [8]:
showElementResults(n, name="normal")
showElementResults(v_surf, name="v on surface")
showElementResults(v_volu, name="v in volume")
showElementResults(v_surf ⋅ n, name="normal flux")
showElementResults(∇ ⋅ v_volu, name="divergence")

openPostProcessor()

-------------------------------------------------------
Version       : 4.15.2-git
License       : GNU General Public License
Build OS      : Linux64-sdk
Build date    : 19700101
Build host    : amdci7.julia.csail.mit.edu
Build options : 64Bit ALGLIB[contrib] ANN[contrib] Bamg Blossom Cairo DIntegration Dlopen DomHex Eigen[contrib] Fltk GMP Gmm[contrib] Hxt Jpeg Kbipack LinuxJoystick MathEx[contrib] Mesh Metis[contrib] Mmg Mpeg Netgen Nii2mesh ONELAB ONELABMetamodel OpenCASCADE OpenCASCADE-CAF OpenGL OpenMP OptHom Parser Plugins Png Post QuadMeshingTools QuadTri Solver TetGen/BR TinyXML2[contrib] Untangle Voro++[contrib] WinslowUntangler Zlib tinyobjloader
FLTK version  : 1.3.8
OCC version   : 7.9.2
Packaged by   : root
Web site      : https://gmsh.info
Issue tracker : https://gitlab.onelab.info/gmsh/gmsh/issues
-------------------------------------------------------


XOpenIM() failed
Fontconfig warning: using without calling FcInit()


## Finalize Gmsh

Close the Gmsh API after finishing the calculations and visualization.

In [9]:
gmsh.finalize()